In [1]:
import os
import json
import numpy as np
from scipy.stats import t, norm

In [2]:
def calcular_kpis_promedio_con_ic_en_formato_json(directorio, nivel_confianza=95, guardar_en=None):
    """
    Calcula el promedio ± IC de todos los KPIs numéricos en archivos JSON de un directorio.
    Usa t-Student si n < 100, o normal estándar (z) si n >= 100.

    Args:
        directorio (str): Ruta a carpeta con archivos JSON.
        nivel_confianza (int or float): Nivel de confianza deseado (ej. 90, 95, 99).
        guardar_en (str or None): Ruta para guardar el resultado en un JSON (opcional).

    Returns:
        dict: Diccionario anidado con 'media ± error' y metadata de cálculo.
    """
    if not (50 <= nivel_confianza < 100):
        raise ValueError("El nivel de confianza debe estar entre 50 y 99.9")

    prob = nivel_confianza / 100
    json_files = [os.path.join(directorio, f) for f in os.listdir(directorio) if f.endswith(".json")]
    n = len(json_files)
    if n == 0:
        raise ValueError("No se encontraron archivos JSON en el directorio.")

    # Usar el primer archivo como referencia de estructura
    with open(json_files[0], "r") as f:
        ejemplo = json.load(f)

    def extraer_rutas(d, prefijo=""):
        rutas = []
        for k, v in d.items():
            ruta = f"{prefijo}.{k}" if prefijo else k
            if isinstance(v, dict):
                rutas += extraer_rutas(v, ruta)
            elif isinstance(v, (int, float)):
                rutas.append(ruta)
        return rutas

    kpis_ruta = extraer_rutas(ejemplo)

    # Recolectar valores de cada KPI
    data = {kpi: [] for kpi in kpis_ruta}
    for file in json_files:
        with open(file, "r") as f:
            contenido = json.load(f)
            for kpi in kpis_ruta:
                try:
                    val = contenido
                    for key in kpi.split("."):
                        val = val[key]
                    if isinstance(val, (int, float)):
                        data[kpi].append(val)
                except (KeyError, TypeError):
                    continue

    resumen = {"_info": {"n": n, "nivel_confianza": f"{nivel_confianza}%", "distribucion": ""}}

    if n >= 100:
        z_val = norm.ppf((1 + prob) / 2)
        resumen["_info"]["distribucion"] = "normal"
    else:
        resumen["_info"]["distribucion"] = "t_student"

    for kpi, valores in data.items():
        arr = np.array(valores)
        mean = np.mean(arr)
        std = np.std(arr, ddof=1)
        sem = std / np.sqrt(n)
        if n >= 100:
            error = z_val * sem
        else:
            t_val = t.ppf((1 + prob) / 2, df=n - 1)
            error = t_val * sem
        valor_str = f"{round(mean, 2)} ± {round(error, 2)}"

        puntero = resumen
        keys = kpi.split(".")
        for key in keys[:-1]:
            puntero = puntero.setdefault(key, {})
        puntero[keys[-1]] = valor_str

    if guardar_en:
        with open(guardar_en, "w") as f:
            json.dump(resumen, f, indent=4)

    return resumen

In [3]:
resumen = calcular_kpis_promedio_con_ic_en_formato_json("resultados simulacion/ModeloProactivo_None_T4500_C4208/kpis", nivel_confianza=95, guardar_en=None)
display(resumen)

{'_info': {'n': 10, 'nivel_confianza': '95%', 'distribucion': 't_student'},
 'LOS_hospitalizado': {'por_hospital_y_unidad': {'Hospital_1': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '0.8 ± 0.07', 'tratamiento': '70.32 ± 0.66'},
    'OR': {'espera': '0.5 ± 0.05', 'tratamiento': '13.01 ± 0.04'},
    'SDU_WARD': {'espera': '0.02 ± 0.0', 'tratamiento': '170.1 ± 0.48'}},
   'Hospital_2': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '0.56 ± 0.04', 'tratamiento': '71.09 ± 0.66'},
    'OR': {'espera': '0.38 ± 0.03', 'tratamiento': '13.07 ± 0.03'},
    'SDU_WARD': {'espera': '0.01 ± 0.0', 'tratamiento': '162.79 ± 0.63'}},
   'Hospital_3': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '0.51 ± 0.05', 'tratamiento': '66.42 ± 0.31'},
    'OR': {'espera': '0.26 ± 0.02', 'tratamiento': '12.97 ± 0.03'},
    'SDU_WARD': {'espera': '0.0 ± 0.0', 'tratamiento': '160.11 ± 0.49'}}},
  'promedio_por_hospital': {'Hospital_1': '229.63 ± 0.68',
   'Ho

In [4]:
resumen = calcular_kpis_promedio_con_ic_en_formato_json("resultados simulacion/ModeloA_None_T4500_C4208/kpis", nivel_confianza=95, guardar_en=None)
display(resumen)

{'_info': {'n': 10, 'nivel_confianza': '95%', 'distribucion': 't_student'},
 'LOS_hospitalizado': {'por_hospital_y_unidad': {'Hospital_1': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '0.8 ± 0.06', 'tratamiento': '70.4 ± 0.57'},
    'OR': {'espera': '0.49 ± 0.04', 'tratamiento': '13.02 ± 0.05'},
    'SDU_WARD': {'espera': '0.02 ± 0.0', 'tratamiento': '169.9 ± 0.39'}},
   'Hospital_2': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '0.55 ± 0.04', 'tratamiento': '71.13 ± 0.55'},
    'OR': {'espera': '0.38 ± 0.02', 'tratamiento': '13.06 ± 0.02'},
    'SDU_WARD': {'espera': '0.01 ± 0.0', 'tratamiento': '162.74 ± 0.61'}},
   'Hospital_3': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '0.51 ± 0.05', 'tratamiento': '66.54 ± 0.18'},
    'OR': {'espera': '0.26 ± 0.03', 'tratamiento': '12.97 ± 0.03'},
    'SDU_WARD': {'espera': '0.01 ± 0.0', 'tratamiento': '159.88 ± 0.35'}}},
  'promedio_por_hospital': {'Hospital_1': '229.68 ± 0.51',
   'H

In [5]:
resumen = calcular_kpis_promedio_con_ic_en_formato_json("resultados sensibilidad/ModeloA_None_T4500_C4208_H1_SDU_WARD_+1/kpis", nivel_confianza=95, guardar_en=None)
display(resumen)

{'_info': {'n': 22, 'nivel_confianza': '95%', 'distribucion': 't_student'},
 'LOS_hospitalizado': {'por_hospital_y_unidad': {'Hospital_1': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '0.77 ± 0.03', 'tratamiento': '70.61 ± 0.28'},
    'OR': {'espera': '0.49 ± 0.02', 'tratamiento': '13.01 ± 0.03'},
    'SDU_WARD': {'espera': '0.02 ± 0.0', 'tratamiento': '169.96 ± 0.29'}},
   'Hospital_2': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '0.54 ± 0.02', 'tratamiento': '71.02 ± 0.24'},
    'OR': {'espera': '0.37 ± 0.02', 'tratamiento': '13.07 ± 0.02'},
    'SDU_WARD': {'espera': '0.01 ± 0.0', 'tratamiento': '162.58 ± 0.29'}},
   'Hospital_3': {'ED': '0.0 ± 0.0',
    'GA': '0.0 ± 0.0',
    'ICU': {'espera': '0.48 ± 0.03', 'tratamiento': '66.54 ± 0.19'},
    'OR': {'espera': '0.24 ± 0.02', 'tratamiento': '12.99 ± 0.02'},
    'SDU_WARD': {'espera': '0.01 ± 0.0', 'tratamiento': '159.79 ± 0.24'}}},
  'promedio_por_hospital': {'Hospital_1': '229.89 ± 0.45',
  

In [ ]:
"""
"costo_diario_promedio": {"General": {
"social_en_wl": 5952.24,
"social_en_hospitales": 12766.53,
"social": 18718.76,
"derivaciones_wl": 2477.6,
"derivaciones_ed": 697.25,
"traslados": 116.67,
"operativo": 3291.53,
"total": 22010.29},

'costo_diario_promedio': {'General': {'social_en_wl': '77.26 ± 14.26',
'social_en_hospitales': '122.14 ± 7.77',
'social': '199.4 ± 20.02',
'derivaciones_wl': '25.94 ± 4.09',
'derivaciones_ed': '842.45 ± 64.65',
'traslados': '93.73 ± 2.72',
'operativo': '962.13 ± 67.23',
'total': '1161.53 ± 81.86'},
"""

'\n"costo_diario_promedio": {"General": {\n"social_en_wl": 5952.24,\n"social_en_hospitales": 12766.53,\n"social": 18718.76,\n"derivaciones_wl": 2477.6,\n"derivaciones_ed": 697.25,\n"traslados": 116.67,\n"operativo": 3291.53,\n"total": 22010.29},\n\n\'costo_diario_promedio\': {\'General\': {\'social_en_wl\': \'97.61 ± 27.75\',\n\'social_en_hospitales\': \'113.98 ± 6.73\',\n\'social\': \'211.58 ± 31.32\',\n\'derivaciones_wl\': \'501.65 ± 4.93\',\n\'derivaciones_ed\': \'709.46 ± 58.89\',\n\'traslados\': \'91.98 ± 2.48\',\n\'operativo\': \'1303.08 ± 62.71\',\n\'total\': \'1514.66 ± 76.16\'},\n\n\'costo_diario_promedio\': {\'General\': {\'social_en_wl\': \'74.73 ± 17.48\',\n\'social_en_hospitales\': \'123.69 ± 9.62\',\n\'social\': \'198.41 ± 25.57\',\n\'derivaciones_wl\': \'0.0 ± 0.0\',\n\'derivaciones_ed\': \'859.89 ± 70.93\',\n\'traslados\': \'93.4 ± 2.26\',\n\'operativo\': \'953.29 ± 71.77\',\n\'total\': \'1151.7 ± 91.3\'},\n\n\'costo_diario_promedio\': {\'General\': {\'social_en_wl\': \